# Clase 9 — VWAP Dinámico + Cierre de Ciclo

**Prerequisito:** L8 VWAP Volume Baselines  
**Duración demo:** ~15 min

---

En L8 construimos un schedule basado en el perfil histórico medio.  
**El problema:** ese schedule se fija *antes* de empezar a ejecutar.

Si el lunes de hoy es un lunes inusualmente activo, te quedas corto en la apertura  
y tienes que recuperar el ritmo más tarde a peores precios.

> **Pregunta:** ¿podemos usar el volumen visto en los primeros 25 minutos  
> para corregir el schedule del resto del día?

Esta clase tiene dos actos:
1. **VWAP Dinámico** — corrección multiplicativa con ventana de 5 intervalos
2. **El Cuadro de Mando** — integrar las señales de L6/L7/L8/L9 en una decisión ejecutable

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.style.use('dark_background')
plt.rcParams.update({'font.family': 'monospace', 'axes.facecolor': '#18181b',
                     'figure.facecolor': '#09090b', 'axes.edgecolor': '#27272a',
                     'grid.color': '#27272a', 'text.color': '#e4e4e7'})

CYAN   = '#22d3ee'
GREEN  = '#4ade80'
RED    = '#f87171'
AMBER  = '#f59e0b'
PURPLE = '#a78bfa'
MUTED  = '#a1a1aa'

TOTAL_QTY = 10.0   # BTC a ejecutar
WINDOW    = 5      # intervalos por bloque (5 × 5 min = 25 min)
TEST_DATE = '2025-12-21'  # día de evaluación

In [ ]:
# Carga de datos (reusa el dataset de L8)
df = pd.read_csv('../08-vwap-volume-baselines/data/btc_volume_intraday.csv',
                 parse_dates=['datetime'])

train_df = df[df['date'] < TEST_DATE]
test_df  = df[df['date'] == TEST_DATE]

# Perfil estático: media de los 20 días anteriores
static_profile = train_df.groupby('interval_idx')['volume_normalized'].mean()
actual_day21   = test_df.sort_values('interval_idx')['volume_normalized'].values

print(f'Train: {train_df["date"].nunique()} días  |  Test: {TEST_DATE}')
print(f'Perfil estático — intervalo 0:   {static_profile.iloc[0]:.6f}')
print(f'Perfil estático — intervalo 144: {static_profile.iloc[144]:.6f}')

## La corrección dinámica

El perfil histórico predice el **promedio**. El día de hoy puede ser diferente.

**Señal:** si los primeros 25 min tuvieron 1.3× el volumen predicho → el resto del día  
probablemente también será más activo → ajusta el schedule hacia adelante.

```python
# Cada WINDOW intervalos (25 min):
cf = realized_vol_last_5 / predicted_vol_last_5
schedule_remaining *= cf
```

- `cf > 1` → mercado más activo → acepta más cantidad ahora
- `cf < 1` → mercado más tranquilo → reduce carga, espera
- `cf ≈ 1` → no hay corrección

In [ ]:
def compute_correction_factor(realized_block: np.ndarray,
                               predicted_block: np.ndarray) -> float:
    """
    Ratio entre volumen realizado y predicho en los últimos WINDOW intervalos.
    Retorna 1.0 si predicted_block.sum() == 0 (evitar división por cero).
    """
    pred_sum = predicted_block.sum()
    if pred_sum == 0:
        return 1.0
    return float(realized_block.sum() / pred_sum)

# Demo: bloque 0 del día 21 (intervalos 0-4, primeros 25 min)
cf_0 = compute_correction_factor(actual_day21[0:5], static_profile.values[0:5])

print(f'Correction factor bloque 0 (00:00–00:20):')
print(f'  Predicho:     {static_profile.values[0:5].sum():.6f}')
print(f'  Realizado:    {actual_day21[0:5].sum():.6f}')
print(f'  CF:           {cf_0:.6f}')
print(f'  El mercado fue {cf_0:.3f}× {"más" if cf_0>1 else "menos"} activo de lo previsto')

In [ ]:
def walk_forward_dynamic(actual: np.ndarray,
                          base_profile: np.ndarray,
                          window: int = 5) -> tuple:
    """
    Simula ejecución dinámica walk-forward.
    Cada `window` intervalos observa lo realizado y corrige el resto del schedule.
    
    Retorna:
      - dynamic_schedule: array con el schedule aplicado intervalo a intervalo
      - cf_history:       lista de correction_factors por bloque
    """
    n = len(actual)
    schedule = base_profile.copy()
    dynamic_schedule = np.zeros(n)
    cf_history = []
    
    for j in range(0, n, window):
        # Ejecuta el bloque actual según el schedule corriente
        dynamic_schedule[j:j+window] = schedule[j:j+window]
        
        # Al final del bloque, calcula CF y corrige el resto
        if j + window < n:
            cf = compute_correction_factor(actual[j:j+window], schedule[j:j+window])
            cf_history.append(cf)
            schedule[j+window:] *= cf
    
    return dynamic_schedule, cf_history

dyn_sched, cf_hist = walk_forward_dynamic(actual_day21, static_profile.values)

print(f'Walk-forward: {len(cf_hist)} bloques de corrección')
print(f'CF mín: {min(cf_hist):.4f}  |  CF máx: {max(cf_hist):.4f}  |  CF medio: {np.mean(cf_hist):.4f}')

In [ ]:
# Visualización: estático vs dinámico vs real — primeras 6 horas
T = 72  # 6 horas
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

ax1.plot(range(T), actual_day21[:T], color=CYAN, linewidth=2, label='Volumen real')
ax1.plot(range(T), static_profile.values[:T], color=MUTED, linewidth=1.5,
         linestyle='--', label='Estático (mean_all)')
ax1.plot(range(T), dyn_sched[:T], color=AMBER, linewidth=2, label='Dinámico (CF)')

for i, (block_start, cf) in enumerate(zip(range(0, T, WINDOW), cf_hist[:T//WINDOW])):
    ax1.axvline(block_start, color=GREEN if cf > 1 else RED, alpha=0.25, linewidth=1)

ax1.set_ylabel('Frac. volumen diario', color=MUTED)
ax1.set_title('Perfil de volumen — primeras 6h del día 21', color=CYAN, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# CFs por bloque
cf_colors = [GREEN if cf > 1 else RED for cf in cf_hist[:T//WINDOW]]
ax2.bar(range(len(cf_hist[:T//WINDOW])), cf_hist[:T//WINDOW],
        color=cf_colors, alpha=0.8, width=0.6)
ax2.axhline(1.0, color=MUTED, linewidth=1.5, linestyle='--', label='CF = 1.0')
ax2.set_xlabel('Bloque (cada 25 min)', color=MUTED)
ax2.set_ylabel('Correction Factor', color=MUTED)
ax2.set_title('Factores de corrección por bloque', color=AMBER, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## La métrica correcta: tracking deviation

El **RMSE del perfil** mide qué tan bien predices el futuro.  
Para *ejecución*, la pregunta es diferente:

> ¿Qué tan cerca estuviste del VWAP ideal en cada momento del día?

**Tracking deviation** = distancia acumulada entre tu ejecución y el ideal en BTC:

- Si sobre-ejecutas en la apertura y el precio baja → pagaste de más
- Si bajo-ejecutas y el precio sube → dejaste de aprovechar la apertura barata

El modelo dinámico no predice mejor el perfil, pero **mantiene el track** mejor.

In [ ]:
# Tracking deviation: desviación acumulada respecto al VWAP ideal
cum_target  = np.cumsum(actual_day21) * TOTAL_QTY
cum_static  = np.cumsum(static_profile.values) * TOTAL_QTY
cum_dynamic = np.cumsum(dyn_sched / dyn_sched.sum()) * TOTAL_QTY

dev_static  = np.abs(cum_static - cum_target)
dev_dynamic = np.abs(cum_dynamic - cum_target)

max_dev_static  = dev_static.max()
max_dev_dynamic = dev_dynamic.max()
improvement     = (max_dev_static - max_dev_dynamic) / max_dev_static * 100

print(f'Tracking deviation máxima (10 BTC, día 21):')
print(f'  Estático:  {max_dev_static:.4f} BTC')
print(f'  Dinámico:  {max_dev_dynamic:.4f} BTC')
print(f'  Mejora:    {improvement:.1f}%')

fig, ax = plt.subplots(figsize=(13, 4))
ax.fill_between(range(288), dev_static * 1000, alpha=0.35, color=RED, label=f'Estático (max={max_dev_static*1000:.0f} mBTC)')
ax.fill_between(range(288), dev_dynamic * 1000, alpha=0.6, color=GREEN, label=f'Dinámico (max={max_dev_dynamic*1000:.0f} mBTC)')

xticks = [0, 72, 144, 216, 287]
ax.set_xticks(xticks)
ax.set_xticklabels(['00:00', '06:00', '12:00', '18:00', '23:55'])
ax.set_xlabel('Hora (UTC)', color=MUTED)
ax.set_ylabel('Tracking deviation (mBTC)', color=MUTED)
ax.set_title(f'Tracking deviation — Estático vs Dinámico (día 21) | mejora {improvement:.0f}%',
             color=CYAN, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Walk-forward backtest completo — días 6 a 21
dates_sorted = sorted(df['date'].unique())
results = []

for i, test_date in enumerate(dates_sorted):
    if i < 5:
        continue
    train_d = df[df['date'] < test_date]
    test_d  = df[df['date'] == test_date]
    sp = train_d.groupby('interval_idx')['volume_normalized'].mean().values
    act = test_d.sort_values('interval_idx')['volume_normalized'].values
    
    dyn, _ = walk_forward_dynamic(act, sp.copy())
    
    ct = np.cumsum(act) * TOTAL_QTY
    cs = np.cumsum(sp) * TOTAL_QTY
    cd = np.cumsum(dyn / dyn.sum()) * TOTAL_QTY
    
    mds = np.abs(cs - ct).max()
    mdd = np.abs(cd - ct).max()
    results.append({'date': test_date, 'static': mds, 'dynamic': mdd, 'dyn_wins': mdd < mds})

rdf = pd.DataFrame(results)
print(f'Promedio estático:  {rdf["static"].mean():.4f} BTC')
print(f'Promedio dinámico:  {rdf["dynamic"].mean():.4f} BTC')
print(f'Días donde gana dinámico: {rdf["dyn_wins"].sum()} / {len(rdf)}')
print(f'Mejora media: {(rdf["static"].mean()-rdf["dynamic"].mean())/rdf["static"].mean()*100:.1f}%')

In [ ]:
# Visualización backtest — side by side por día
fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(rdf))
w = 0.35

ax.bar(x - w/2, rdf['static'] * 1000, w, color=RED, alpha=0.8, label='Estático')
ax.bar(x + w/2, rdf['dynamic'] * 1000, w, color=GREEN, alpha=0.8, label='Dinámico')

ax.set_xticks(x)
ax.set_xticklabels([str(d)[5:] for d in rdf['date']], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Max tracking deviation (mBTC)', color=MUTED)
ax.set_title('Walk-forward backtest — 16 días de evaluación', color=CYAN, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
# Acto 2 — El Cuadro de Mando

En las últimas 4 clases construimos 3 señales independientes:

| Señal | Clase | Qué mide |
|-------|-------|----------|
| `imbalance_mean_5` | L6/L7 | Presión compradora vs vendedora en el LOB |
| `fill_probability` | L7 | Probabilidad de que un limit order se ejecute en 3 snapshots |
| `volume_ratio` | L8/L9 | Volumen real ÷ volumen predicho (últimos 5 min) |

¿Cómo combinarlas en una **decisión ejecutable**?

Construimos `ExecutionDecision` — el árbitro que conecta L2 (OOP) con L6-L9.

In [ ]:
class ExecutionDecision:
    """
    Combina señales del LOB y del volumen para decidir el tipo de orden.
    
    Parámetros
    ----------
    imbalance    : float [0,1]  — fracción bid/(bid+ask) del LOB (L6/L7)
    fill_prob    : float [0,1]  — probabilidad de fill en 3 intervalos (L7)
    volume_ratio : float >0     — vol realizado / vol predicho últimos 5 min (L8/L9)
    """
    IMBALANCE_BULL = 0.55  # > 0.55 = presión compradora
    FILL_MIN       = 0.50  # > 0.50 = alta prob de fill
    VOL_HIGH       = 1.20  # > 1.20 = día activo (ventana de oportunidad)
    VOL_LOW        = 0.80  # < 0.80 = día tranquilo (esperar)
    
    def __init__(self, imbalance: float, fill_prob: float, volume_ratio: float):
        self.imbalance    = imbalance
        self.fill_prob    = fill_prob
        self.volume_ratio = volume_ratio
    
    def decide(self) -> str:
        """
        Retorna 'LIMIT', 'MARKET' o 'WAIT'.
        
        LIMIT:  señal alcista + alta prob fill → orden pasiva beneficiosa
        MARKET: volumen alto + señal no alcista → ejecutar activamente ahora
        WAIT:   volumen bajo o señal neutral → conservar munición
        """
        if self.imbalance > self.IMBALANCE_BULL and self.fill_prob > self.FILL_MIN:
            return 'LIMIT'
        elif (self.volume_ratio > self.VOL_HIGH and
              (self.imbalance < (1 - self.IMBALANCE_BULL) or self.fill_prob <= self.FILL_MIN)):
            return 'MARKET'
        else:
            return 'WAIT'

In [ ]:
# Demo — 4 escenarios canónicos
scenarios = [
    (0.62, 0.65, 1.35, 'LOB alcista + fill alto + vol activo'),
    (0.38, 0.35, 1.55, 'LOB bajista + fill bajo + vol alto'),
    (0.50, 0.45, 0.65, 'LOB neutro + fill incierto + vol bajo'),
    (0.60, 0.51, 0.90, 'LOB alcista + fill ok + vol moderado'),
]

print(f'{"Imb":>5} {"Fill":>5} {"VolR":>5} | {"Decisión":<8} | Situación')
print('-' * 70)
for imb, fp, vr, desc in scenarios:
    d = ExecutionDecision(imb, fp, vr)
    decision = d.decide()
    icon = {'LIMIT': '🟢', 'MARKET': '🔴', 'WAIT': '🟡'}.get(decision, ' ')
    print(f'{imb:>5.2f} {fp:>5.2f} {vr:>5.2f} | {icon} {decision:<6} | {desc}')

In [ ]:
# Scan sobre datos reales del LOB (L7)
# NOTA: LOB data y volume data tienen períodos distintos — NO los unimos.
# Usamos vol_ratio=1.0 para simular el estado sin corrección de volumen.
lob_df = pd.read_csv('../07-lob-modeling-examples/data/lob_modeling_features.csv')

# Usar RF fill probabilities simuladas (los datos L7 tienen fill_in_3 como binario)
# Para la demo usamos fill_prob = 0.5 + 0.2 * (2*fill_in_3 - 1) como proxy
decisions = []
for _, row in lob_df.iterrows():
    fill_proxy = 0.5 + 0.2 * (2 * row['fill_in_3'] - 1)
    d = ExecutionDecision(
        imbalance    = row['imbalance_mean_5'],
        fill_prob    = fill_proxy,
        volume_ratio = 1.0  # sin señal de volumen
    )
    decisions.append(d.decide())

decision_counts = pd.Series(decisions).value_counts()
print('Distribución de decisiones sobre 484 snapshots del LOB real:')
for dec, cnt in decision_counts.items():
    print(f'  {dec:<8}: {cnt:3d} ({cnt/len(decisions)*100:.1f}%)')

In [ ]:
# Visualización del cuadro de mando — distribución de decisiones
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart de decisiones
colors_pie = {'LIMIT': GREEN, 'MARKET': RED, 'WAIT': AMBER}
labels = list(decision_counts.index)
vals   = list(decision_counts.values)
cols   = [colors_pie.get(l, MUTED) for l in labels]
ax1.pie(vals, labels=[f'{l}\n{v/sum(vals)*100:.0f}%' for l, v in zip(labels, vals)],
        colors=cols, startangle=90, wedgeprops={'edgecolor': '#09090b', 'linewidth': 2})
ax1.set_title('Distribución de decisiones\n(484 snapshots LOB, vol_ratio=1.0)', color=CYAN, fontweight='bold')

# Scatter imbalance_mean_5 vs decisión
colors_dec = {'LIMIT': GREEN, 'MARKET': RED, 'WAIT': AMBER}
for dec in ['LIMIT', 'MARKET', 'WAIT']:
    mask = [d == dec for d in decisions]
    imb_vals = lob_df['imbalance_mean_5'].values[mask]
    ax2.scatter(range(sum(mask)), sorted(imb_vals), color=colors_dec[dec],
                alpha=0.4, s=8, label=dec)
ax2.axhline(ExecutionDecision.IMBALANCE_BULL, color=CYAN, linewidth=1.5, linestyle='--',
            label=f'Bull threshold ({ExecutionDecision.IMBALANCE_BULL})')
ax2.set_xlabel('Snapshots ordenados por imbalance', color=MUTED)
ax2.set_ylabel('imbalance_mean_5', color=MUTED)
ax2.set_title('imbalance_mean_5 por tipo de decisión', color=CYAN, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Cierre de Ciclo: L4 → L9

| Clase | Pregunta central | Artefacto clave |
|-------|-----------------|----------------|
| **L4** | ¿Cómo se estructura el LOB? | `btc_lob_snapshots.csv` + visualización bid/ask |
| **L5** | ¿Cuándo se ejecuta una orden límite? | Fill probability empírica |
| **L6** | ¿Puedes predecir la dirección del precio? | Pipeline ML + data leakage demo |
| **L7** | ¿Qué modelo funciona mejor? | `DecisionTree` overfitting + `RandomForest` + fill target |
| **L8** | ¿Cuándo hay liquidez intradiaria? | `build_vwap_schedule(total_qty, profile)` |
| **L9** | ¿Cómo adaptamos el schedule en tiempo real? | `walk_forward_dynamic()` + `ExecutionDecision` |

---

## Limitaciones

- Los datos son **sintéticos** — patrones más limpios de lo real
- El cuadro de mando usa `volume_ratio=1.0` para LOB snapshots (períodos distintos)
- `ExecutionDecision` es una heurística, no un modelo calibrado
- No hay costes de transacción, impacto de mercado dinámico, ni latencia
- Con más datos (meses en lugar de semanas), los perfiles serían más estables

## Puente al Exam-Quiz I (L10)

En **L10** tenemos el **Exam-Quiz I** — cierre del primer bloque del curso (L1–L9).  
El examen cubre **conceptos, código y razonamiento** sobre todo lo construido.  

Repasa especialmente:
- `build_vwap_schedule` y la idea de perfil normalizado (L8)
- Qué hace `compute_correction_factor` y por qué mejora el tracking (L9)
- Las 3 señales de `ExecutionDecision` y qué clase las originó (L6-L9)
- La diferencia entre overfitting (L7), data leakage (L6) y tracking deviation (L9)